# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


Este capítulo depende das tabelas 00–06. Se estiverem ausentes, a célula seguinte executa as etapas prévias. Ao usar o master, não há recálculo.

In [2]:
required_outputs = ['data_audit', 'channel_summary', 'within_between_variance', 'geo_time_r2_per_capita', 'vif_summary', 'media_mix', 'meridian_eda_checks', 'meridian_data_adequacy', 'outliers']
if any(not (ROOT / 'outputs/tables' / f'{name}.csv').exists() for name in required_outputs):
    from src.eda import data_audit, general, temporal, relationships
    from src.geo_analysis import geo_analysis, media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit, general, temporal, geo_analysis, media_geo, relationships, official_eda]:
        analysis(geo_time_panel)


## 07 — Há evidência para justificar investigar um MMM geo-level?

### Pergunta
Há evidência para justificar investigar um MMM geo-level?

### Motivação
Precisamos responder às perguntas do TCC com resultados, limites e implicações, evitando transformar descrição em causalidade.

### Método
Consolidar integridade, missing, extremos, variabilidade, população, mix, within/between, R², colinearidade e adequação oficial; produzir relatório de prontidão e narrativa final.

### Interpretação antes de olhar os resultados
Informação geo-temporal residual pode justificar a investigação hierárquica, mas não garante causalidade, generalização ou superioridade preditiva. A próxima fase requer especificação, priors, holdout e diagnósticos posteriores.

**Dependências:** painel validado e funções em `src/reporting.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.reporting import conclusions
report_07 = conclusions(geo_time_panel)
show(report_07)

### Resultado observado e conclusão parcial

1. **Estrutura.** O painel tem 40 geos × 156 semanas = 6240 observações, completude de 100.0% e 0 células ausentes. A chave geo–tempo foi validada.

2. **Tempo.** Todos os 5 canais pagos variam; a parcela within-geo da variância fica entre 61.0% e 94.7%. Channel2 é o canal com mais zeros: 64.3% das células, com 5.5% do gasto. A tabela de atividade e os gráficos distinguem semanas sem mídia de mudanças de intensidade. Movimentos de nível e sazonalidade são exploratórios.

3. **Geografia.** As médias por geo explicam de 5.3% a 39.0% da variância bruta da mídia paga. Rankings de volume dependem de população; mix e medidas por habitante oferecem lentes complementares.

4. **GEO × TEMPO.** Após retirar médias de geo e de semana, permanece de 42.9% a 77.9% da variância bruta de mídia paga; por habitante, de 64.8% a 79.3%. Esse componente é perdido ao observar apenas o agregado nacional. O painel contém variação potencialmente informativa, mas o resíduo pode conter ruído, endogeneidade e confundimento. Comparar com a decomposição per capita é essencial: diferenças multiplicativas de tamanho também podem gerar resíduos aditivos.

5. **Identificabilidade.** R², correlações, VIF e os alertas oficiais devem orientar hipóteses de especificação. Nenhuma dessas métricas prova identificação causal ou que um modelo geo terá melhor previsão que o nacional. Essa comparação exige futura modelagem com validação temporal comum, priors justificados e diagnósticos posteriores.

**Prontidão:** a integridade permite continuar o estudo metodológico, condicionando a próxima fase à revisão dos alertas oficiais e das limitações dos dados simulados. Não há recomendação de alocação de orçamento nesta EDA.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Desenho causal, especificação, priors, holdout temporal e posterior em fase futura.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**readiness_report** — 18 linhas; CSV completo em `outputs/tables/`.

,Dimension,Question,Evidence,Finding,Modeling_implication
0,integridade,Painel completo?,data_audit,"6240 linhas, 40 geos, 156 semanas; completude 1.0",Contrato validado; preservar raw e schema.
1,missing,Há valores ausentes?,missing_summary,0 células ausentes,Não imputar dados completos.
2,outliers,Há extremos descritivos?,outliers,6039 pares observação/variável sinalizados por...,Investigar escala populacional e zeros; não ex...
3,KPI variation,Existe variação?,within_between_variance,1/1 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
4,media variation,Existe variação?,within_between_variance,6/6 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
5,spend variation,Existe variação?,within_between_variance,5/5 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
6,control collinearity,Existe variação?,within_between_variance,2/2 variáveis com variância positiva; resíduo ...,"Variação é necessária, não suficiente para ide..."
7,temporal variation,Quanto varia a mídia paga?,geo_time_r2,r2_time: 13.63% a 19.07%,Avaliar também escala por população e colinear...
8,geo variation,Quanto varia a mídia paga?,geo_time_r2,r2_geo: 5.35% a 38.99%,Avaliar também escala por população e colinear...
9,within variation,Quanto varia a mídia paga?,geo_time_r2,within_share: 61.01% a 94.65%,Avaliar também escala por população e colinear...


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]